
# Chilli Cross-Dataset Results Analysis

**Study:** *Cross-Dataset Generalization of Lightweight Deep Learning for Multi-Class Chilli Leaf Disease Classification*

This notebook reads the experiment outputs produced by:

- `run_baselines.py`
- `run_proposed.py`
- `statistical_analysis.py`

It generates manuscript-ready:

- descriptive tables;
- mean ± standard deviation summaries;
- pairwise cross-dataset comparison tables;
- multi-source held-out-domain tables;
- pooled and within-dataset CV tables;
- parameter/efficiency tables;
- training and validation curves;
- confusion matrices;
- per-class metric plots;
- ROC curves;
- model-rank figures;
- Friedman/Wilcoxon/Holm summaries;
- paired-bootstrap and McNemar outputs when available.

> **Important:** Run this notebook from the project root, where `results/` and `data/` are located.


In [ ]:

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.metrics import (
    roc_curve,
    auc,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
SUMMARY_CSV = RESULTS_DIR / "summary.csv"

FIGURE_DIR = RESULTS_DIR / "manuscript_figures"
TABLE_DIR = RESULTS_DIR / "manuscript_tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Results dir  :", RESULTS_DIR)
print("Figures dir  :", FIGURE_DIR)
print("Tables dir   :", TABLE_DIR)

if not SUMMARY_CSV.exists():
    raise FileNotFoundError(
        f"Missing {SUMMARY_CSV}\n"
        "Run at least one training experiment first."
    )


## 1. Load and inspect all run-level results

In [ ]:

summary = pd.read_csv(SUMMARY_CSV)

print(f"Loaded {len(summary):,} completed runs.")
print("\nExperiments:")
print(summary["experiment"].value_counts(dropna=False))

print("\nModels:")
print(summary["model_name"].value_counts(dropna=False))

display(summary.head())


In [ ]:

# Stable display names for manuscript tables.
MODEL_DISPLAY = {
    "mobilenet_v2": "MobileNetV2",
    "mobilenet_v3_small": "MobileNetV3-Small",
    "shufflenet_v2_x1_0": "ShuffleNetV2",
    "efficientnet_b0": "EfficientNet-B0",
    "efficientnet_b4": "EfficientNet-B4",
    "resnet50": "ResNet50",
    "densenet121": "DenseNet121",
    "swin_t": "Swin-Tiny",
    "chilli_lite_gfnet": "ChilliLite-GFNet",
}

CANONICAL_MODEL_ORDER = [
    "mobilenet_v2",
    "mobilenet_v3_small",
    "shufflenet_v2_x1_0",
    "efficientnet_b0",
    "efficientnet_b4",
    "resnet50",
    "densenet121",
    "swin_t",
    "chilli_lite_gfnet",
]

summary["model_display"] = summary["model_name"].map(MODEL_DISPLAY).fillna(summary["model_name"])

metric_cols = [c for c in summary.columns if c.startswith("test_")]
print("Available test metrics:")
print(metric_cols)


## 2. Data completeness and experiment coverage

In [ ]:

coverage = (
    summary.groupby(["experiment", "scenario", "model_display"])
    .size()
    .rename("completed_runs")
    .reset_index()
    .sort_values(["experiment", "scenario", "model_display"])
)

display(coverage)

coverage.to_csv(TABLE_DIR / "experiment_coverage.csv", index=False)
print("Saved:", TABLE_DIR / "experiment_coverage.csv")



## 3. Generic descriptive summary

For each scenario and model, the notebook reports:

- number of completed runs;
- mean;
- standard deviation;
- median;
- 95% confidence interval based on the Student-\(t\) distribution.

For the manuscript, **mean ± SD** is the primary summary for repeated folds/seeds.


In [ ]:

try:
    from scipy.stats import t
except ImportError as exc:
    raise ImportError("Install SciPy: pip install scipy") from exc

def descriptive_summary(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    rows = []
    for (experiment, scenario, model), g in df.groupby(
        ["experiment", "scenario", "model_display"]
    ):
        values = g[metric].dropna().astype(float).to_numpy()
        if len(values) == 0:
            continue

        n = len(values)
        mean = float(values.mean())
        std = float(values.std(ddof=1)) if n > 1 else np.nan
        sem = std / math.sqrt(n) if n > 1 else np.nan
        ci_half = (
            float(t.ppf(0.975, n - 1) * sem)
            if n > 1 and np.isfinite(sem)
            else np.nan
        )

        rows.append({
            "experiment": experiment,
            "scenario": scenario,
            "model": model,
            "n": n,
            "mean": mean,
            "std": std,
            "median": float(np.median(values)),
            "ci95_low": mean - ci_half if np.isfinite(ci_half) else np.nan,
            "ci95_high": mean + ci_half if np.isfinite(ci_half) else np.nan,
        })

    return pd.DataFrame(rows)

macro_f1_desc = descriptive_summary(summary, "test_macro_f1")
display(macro_f1_desc.head(20))

macro_f1_desc.to_csv(TABLE_DIR / "macro_f1_descriptive_summary.csv", index=False)


## 4. Helper functions for manuscript tables

In [ ]:

def mean_std_text(values, digits=4):
    values = pd.Series(values).dropna().astype(float)
    if len(values) == 0:
        return "--"
    if len(values) == 1:
        return f"{values.iloc[0]:.{digits}f}"
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


def make_metric_table(
    df: pd.DataFrame,
    experiment: str,
    metrics=("test_accuracy", "test_macro_f1", "test_mcc", "test_auc_ovr_macro"),
    digits=4,
):
    x = df[df["experiment"] == experiment].copy()
    if x.empty:
        return pd.DataFrame()

    rows = []
    for (scenario, model), g in x.groupby(["scenario", "model_display"]):
        row = {
            "Scenario": scenario,
            "Model": model,
            "Runs": len(g),
        }
        for metric in metrics:
            if metric in g.columns:
                row[metric] = mean_std_text(g[metric], digits=digits)
        if "parameters" in g.columns:
            row["Parameters (M)"] = g["parameters"].dropna().mean() / 1e6
        rows.append(row)

    return pd.DataFrame(rows).sort_values(["Scenario", "Model"]).reset_index(drop=True)


def save_table(df: pd.DataFrame, filename: str):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path


## 5. Within-dataset 5-fold CV table

In [ ]:

within_table = make_metric_table(summary, "within_cv")
if within_table.empty:
    print("No within_cv results found yet.")
else:
    display(within_table)
    save_table(within_table, "within_cv_results.csv")


## 6. Bidirectional pairwise cross-dataset table

In [ ]:

pairwise_table = make_metric_table(summary, "pairwise")
if pairwise_table.empty:
    print("No pairwise results found yet.")
else:
    display(pairwise_table)
    save_table(pairwise_table, "pairwise_cross_dataset_results.csv")


## 7. Multi-source held-out-domain table

In [ ]:

multisource_table = make_metric_table(summary, "multisource")
if multisource_table.empty:
    print("No multisource results found yet.")
else:
    display(multisource_table)
    save_table(multisource_table, "multisource_results.csv")


## 8. Pooled 5-fold CV table

In [ ]:

pooled_table = make_metric_table(summary, "pooled_cv")
if pooled_table.empty:
    print("No pooled_cv results found yet.")
else:
    display(pooled_table)
    save_table(pooled_table, "pooled_cv_results.csv")


## 9. Best model per scenario

In [ ]:

def best_model_per_scenario(df, experiment, metric="test_macro_f1"):
    x = df[df["experiment"] == experiment].copy()
    if x.empty:
        return pd.DataFrame()

    agg = (
        x.groupby(["scenario", "model_display"])[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    idx = agg.groupby("scenario")["mean"].idxmax()
    best = agg.loc[idx].sort_values("scenario").reset_index(drop=True)
    best = best.rename(columns={
        "model_display": "best_model",
        "mean": f"{metric}_mean",
        "std": f"{metric}_std",
        "count": "runs",
    })
    return best

for exp in ["within_cv", "pairwise", "multisource", "pooled_cv"]:
    best = best_model_per_scenario(summary, exp)
    if not best.empty:
        print(f"\nBest model by Macro-F1: {exp}")
        display(best)
        save_table(best, f"best_model_{exp}.csv")



## 10. Pairwise cross-dataset Macro-F1 figure

This grouped bar chart compares models across the six bidirectional transfer directions.
Error bars show standard deviation when multiple seeds are available.


In [ ]:

def grouped_bar_by_scenario(
    df,
    experiment,
    metric="test_macro_f1",
    ylabel="Macro-F1",
    filename=None,
):
    x = df[df["experiment"] == experiment].copy()
    if x.empty:
        print(f"No {experiment} results found.")
        return

    agg = (
        x.groupby(["scenario", "model_display"])[metric]
        .agg(["mean", "std"])
        .reset_index()
    )

    scenarios = list(agg["scenario"].drop_duplicates())
    models = [m for m in [MODEL_DISPLAY.get(k, k) for k in CANONICAL_MODEL_ORDER]
              if m in set(agg["model_display"])]

    width = 0.8 / max(1, len(models))
    xpos = np.arange(len(scenarios))

    fig, ax = plt.subplots(figsize=(max(10, len(scenarios) * 1.8), 6))

    for i, model in enumerate(models):
        vals = []
        errs = []
        for scenario in scenarios:
            row = agg[
                (agg["scenario"] == scenario)
                & (agg["model_display"] == model)
            ]
            vals.append(row["mean"].iloc[0] if len(row) else np.nan)
            errs.append(row["std"].iloc[0] if len(row) else np.nan)

        ax.bar(
            xpos + (i - (len(models) - 1) / 2) * width,
            vals,
            width=width,
            yerr=errs,
            capsize=2,
            label=model,
        )

    ax.set_xticks(xpos)
    ax.set_xticklabels(scenarios, rotation=30, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Cross-dataset scenario")
    ax.set_title(f"{ylabel} across {experiment} scenarios")
    ax.legend(ncol=2, fontsize=9)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()

    if filename:
        path = FIGURE_DIR / filename
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print("Saved:", path)

    plt.show()


grouped_bar_by_scenario(
    summary,
    experiment="pairwise",
    metric="test_macro_f1",
    ylabel="Macro-F1",
    filename="pairwise_macro_f1.png",
)


## 11. Multi-source Macro-F1 figure

In [ ]:

grouped_bar_by_scenario(
    summary,
    experiment="multisource",
    metric="test_macro_f1",
    ylabel="Macro-F1",
    filename="multisource_macro_f1.png",
)



## 12. Cross-dataset transfer matrix

For a selected model, this figure visualizes directional transfer performance.
Rows are source datasets and columns are target datasets.


In [ ]:

def transfer_matrix(
    df,
    model_name,
    metric="test_macro_f1",
    filename=None,
):
    x = df[
        (df["experiment"] == "pairwise")
        & (df["model_name"] == model_name)
    ].copy()

    if x.empty:
        print(f"No pairwise results for {model_name}.")
        return

    agg = x.groupby("scenario")[metric].mean()

    datasets = ["A", "B", "C"]
    mat = np.full((3, 3), np.nan)

    for i, src in enumerate(datasets):
        for j, tgt in enumerate(datasets):
            if src == tgt:
                continue
            scenario = f"{src}->{tgt}"
            if scenario in agg.index:
                mat[i, j] = agg.loc[scenario]

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(mat)

    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(datasets)
    ax.set_yticklabels(datasets)
    ax.set_xlabel("Target dataset")
    ax.set_ylabel("Source dataset")
    ax.set_title(f"Directional transfer Macro-F1: {MODEL_DISPLAY.get(model_name, model_name)}")

    for i in range(3):
        for j in range(3):
            if np.isfinite(mat[i, j]):
                ax.text(j, i, f"{mat[i, j]:.3f}", ha="center", va="center")

    fig.colorbar(im, ax=ax, label=metric)
    fig.tight_layout()

    if filename:
        path = FIGURE_DIR / filename
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print("Saved:", path)

    plt.show()


# Change this model if needed.
SELECTED_MODEL = (
    "chilli_lite_gfnet"
    if "chilli_lite_gfnet" in set(summary["model_name"])
    else "mobilenet_v2"
)

transfer_matrix(
    summary,
    SELECTED_MODEL,
    filename=f"transfer_matrix_{SELECTED_MODEL}.png",
)


## 13. Accuracy–efficiency trade-off

In [ ]:

def efficiency_scatter(
    df,
    experiment="pairwise",
    metric="test_macro_f1",
    filename=None,
):
    x = df[df["experiment"] == experiment].copy()
    if x.empty:
        print(f"No {experiment} results found.")
        return

    agg = (
        x.groupby(["model_name", "model_display"])
        .agg(
            metric_mean=(metric, "mean"),
            parameters=("parameters", "mean"),
            elapsed_seconds=("elapsed_seconds", "mean"),
        )
        .reset_index()
    )
    agg["parameters_m"] = agg["parameters"] / 1e6

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        agg["parameters_m"],
        agg["metric_mean"],
        s=70,
    )

    for _, row in agg.iterrows():
        ax.annotate(
            row["model_display"],
            (row["parameters_m"], row["metric_mean"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )

    ax.set_xscale("log")
    ax.set_xlabel("Parameters (millions, log scale)")
    ax.set_ylabel("Mean Macro-F1")
    ax.set_title(f"Accuracy–efficiency trade-off ({experiment})")
    ax.grid(alpha=0.25)
    fig.tight_layout()

    if filename:
        path = FIGURE_DIR / filename
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print("Saved:", path)

    plt.show()

    return agg.sort_values("metric_mean", ascending=False)


efficiency_table = efficiency_scatter(
    summary,
    experiment="pairwise",
    metric="test_macro_f1",
    filename="pairwise_accuracy_efficiency.png",
)

if efficiency_table is not None:
    display(efficiency_table)
    efficiency_table.to_csv(TABLE_DIR / "pairwise_efficiency_summary.csv", index=False)



## 14. Select one run for detailed inspection

The next cells inspect:

- `history.csv`
- `confusion_matrix.csv`
- `per_class_metrics.csv`
- `predictions.csv`

Change the selectors below to any completed run.


In [ ]:

# ---------------------------------------------------------------------
# Run selector
# ---------------------------------------------------------------------
DETAIL_EXPERIMENT = "pairwise"
DETAIL_SCENARIO = "A->B"
DETAIL_MODEL = SELECTED_MODEL

detail_candidates = summary[
    (summary["experiment"] == DETAIL_EXPERIMENT)
    & (summary["scenario"] == DETAIL_SCENARIO)
    & (summary["model_name"] == DETAIL_MODEL)
].copy()

if detail_candidates.empty:
    print("Requested detailed run not found.")
    print("Available examples:")
    display(
        summary[
            ["experiment", "scenario", "model_name", "seed", "metrics_path"]
        ].head(20)
    )
    DETAIL_RUN = None
else:
    DETAIL_RUN = detail_candidates.iloc[0]
    display(detail_candidates)
    print("Selected metrics file:", DETAIL_RUN["metrics_path"])


## 15. Training and validation history

In [ ]:

def run_dir_from_summary_row(row):
    return Path(row["metrics_path"]).resolve().parent

if DETAIL_RUN is not None:
    run_dir = run_dir_from_summary_row(DETAIL_RUN)
    history_path = run_dir / "history.csv"

    if history_path.exists():
        history = pd.read_csv(history_path)
        display(history.head())

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(history["epoch"], history["train_loss"], label="Train loss")
        ax.plot(history["epoch"], history["val_loss"], label="Validation loss")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(
            f"Loss convergence: {MODEL_DISPLAY.get(DETAIL_MODEL, DETAIL_MODEL)} "
            f"({DETAIL_SCENARIO})"
        )
        ax.legend()
        ax.grid(alpha=0.25)
        fig.tight_layout()

        path = FIGURE_DIR / f"history_loss_{DETAIL_MODEL}_{DETAIL_SCENARIO.replace('->','_to_')}.png"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print("Saved:", path)
        plt.show()

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(history["epoch"], history["train_macro_f1"], label="Train Macro-F1")
        ax.plot(history["epoch"], history["val_macro_f1"], label="Validation Macro-F1")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Macro-F1")
        ax.set_title(
            f"Macro-F1 convergence: {MODEL_DISPLAY.get(DETAIL_MODEL, DETAIL_MODEL)} "
            f"({DETAIL_SCENARIO})"
        )
        ax.legend()
        ax.grid(alpha=0.25)
        fig.tight_layout()

        path = FIGURE_DIR / f"history_f1_{DETAIL_MODEL}_{DETAIL_SCENARIO.replace('->','_to_')}.png"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print("Saved:", path)
        plt.show()
    else:
        print("Missing:", history_path)


## 16. Confusion matrix

In [ ]:

if DETAIL_RUN is not None:
    run_dir = run_dir_from_summary_row(DETAIL_RUN)
    cm_path = run_dir / "confusion_matrix.csv"

    if cm_path.exists():
        cm_df = pd.read_csv(cm_path, index_col=0)
        display(cm_df)

        cm = cm_df.to_numpy(dtype=float)
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_norm = np.divide(
            cm,
            row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0,
        )

        fig, ax = plt.subplots(figsize=(8, 7))
        im = ax.imshow(cm_norm)

        labels = list(cm_df.index)
        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_yticklabels(labels)
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title(
            f"Normalized confusion matrix: "
            f"{MODEL_DISPLAY.get(DETAIL_MODEL, DETAIL_MODEL)} ({DETAIL_SCENARIO})"
        )

        for i in range(cm_norm.shape[0]):
            for j in range(cm_norm.shape[1]):
                ax.text(
                    j, i,
                    f"{cm_norm[i, j]:.2f}\n({int(cm[i, j])})",
                    ha="center",
                    va="center",
                    fontsize=8,
                )

        fig.colorbar(im, ax=ax, label="Row-normalized proportion")
        fig.tight_layout()

        path = FIGURE_DIR / f"confusion_{DETAIL_MODEL}_{DETAIL_SCENARIO.replace('->','_to_')}.png"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print("Saved:", path)
        plt.show()
    else:
        print("Missing:", cm_path)


## 17. Per-class metrics

In [ ]:

if DETAIL_RUN is not None:
    run_dir = run_dir_from_summary_row(DETAIL_RUN)
    per_class_path = run_dir / "per_class_metrics.csv"

    if per_class_path.exists():
        per_class = pd.read_csv(per_class_path)
        display(per_class)

        metrics_to_plot = [
            "precision",
            "recall_sensitivity",
            "specificity",
            "f1",
        ]
        available = [m for m in metrics_to_plot if m in per_class.columns]

        x = np.arange(len(per_class))
        width = 0.8 / max(1, len(available))

        fig, ax = plt.subplots(figsize=(max(9, len(per_class) * 1.6), 6))

        for i, metric in enumerate(available):
            ax.bar(
                x + (i - (len(available) - 1) / 2) * width,
                per_class[metric],
                width=width,
                label=metric.replace("_", " ").title(),
            )

        ax.set_xticks(x)
        ax.set_xticklabels(per_class["class_name"], rotation=35, ha="right")
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Score")
        ax.set_title(
            f"Per-class performance: "
            f"{MODEL_DISPLAY.get(DETAIL_MODEL, DETAIL_MODEL)} ({DETAIL_SCENARIO})"
        )
        ax.legend()
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()

        path = FIGURE_DIR / f"per_class_{DETAIL_MODEL}_{DETAIL_SCENARIO.replace('->','_to_')}.png"
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print("Saved:", path)
        plt.show()
    else:
        print("Missing:", per_class_path)


## 18. Multi-class ROC curves

In [ ]:

if DETAIL_RUN is not None:
    run_dir = run_dir_from_summary_row(DETAIL_RUN)
    pred_path = run_dir / "predictions.csv"

    if pred_path.exists():
        pred = pd.read_csv(pred_path)
        prob_cols = [c for c in pred.columns if c.startswith("prob_")]

        if not prob_cols:
            print("No probability columns found.")
        else:
            fig, ax = plt.subplots(figsize=(8, 6))

            auc_rows = []
            y_true = pred["y_true"].astype(str)

            for prob_col in prob_cols:
                class_name = prob_col.replace("prob_", "", 1)
                y_bin = (y_true == class_name).astype(int).to_numpy()

                if len(np.unique(y_bin)) < 2:
                    continue

                scores = pred[prob_col].to_numpy(dtype=float)
                fpr, tpr, _ = roc_curve(y_bin, scores)
                auc_value = auc(fpr, tpr)

                ax.plot(fpr, tpr, label=f"{class_name} (AUC={auc_value:.3f})")
                auc_rows.append({
                    "class_name": class_name,
                    "auc": auc_value,
                })

            ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
            ax.set_xlabel("False positive rate")
            ax.set_ylabel("True positive rate")
            ax.set_title(
                f"One-vs-rest ROC curves: "
                f"{MODEL_DISPLAY.get(DETAIL_MODEL, DETAIL_MODEL)} ({DETAIL_SCENARIO})"
            )
            ax.legend(fontsize=8)
            ax.grid(alpha=0.25)
            fig.tight_layout()

            path = FIGURE_DIR / f"roc_{DETAIL_MODEL}_{DETAIL_SCENARIO.replace('->','_to_')}.png"
            fig.savefig(path, dpi=300, bbox_inches="tight")
            print("Saved:", path)
            plt.show()

            auc_table = pd.DataFrame(auc_rows)
            display(auc_table)
    else:
        print("Missing:", pred_path)



## 19. Statistical analysis outputs

This section reads outputs generated by:

```bash
python3 statistical_analysis.py \
  --experiment pairwise \
  --metric test_macro_f1
```

Expected files include:

- `descriptive_summary.csv`
- `average_ranks.csv`
- `friedman_tests.csv`
- `pairwise_wilcoxon_holm.csv`


In [ ]:

STAT_EXPERIMENT = "pairwise"
STAT_DIR = RESULTS_DIR / "statistics" / STAT_EXPERIMENT

if not STAT_DIR.exists():
    print(
        f"Statistics directory not found: {STAT_DIR}\n"
        "Run statistical_analysis.py first."
    )
else:
    for file in sorted(STAT_DIR.glob("*.csv")):
        print(file.name)


### 19.1 Friedman omnibus test

In [ ]:

friedman_path = STAT_DIR / "friedman_tests.csv"

if friedman_path.exists():
    friedman = pd.read_csv(friedman_path)
    display(friedman)
else:
    print("Missing:", friedman_path)


### 19.2 Average model ranks

In [ ]:

ranks_path = STAT_DIR / "average_ranks.csv"

if ranks_path.exists():
    ranks = pd.read_csv(ranks_path)
    display(ranks)

    ranks_plot = ranks.sort_values("average_rank", ascending=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(ranks_plot["model_name"], ranks_plot["average_rank"])
    ax.invert_yaxis()
    ax.set_xlabel("Average rank (lower is better)")
    ax.set_title(f"Average model ranks: {STAT_EXPERIMENT}")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    path = FIGURE_DIR / f"average_ranks_{STAT_EXPERIMENT}.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)
    plt.show()
else:
    print("Missing:", ranks_path)


### 19.3 Pairwise Wilcoxon tests with Holm correction

In [ ]:

wilcoxon_path = STAT_DIR / "pairwise_wilcoxon_holm.csv"

if wilcoxon_path.exists():
    wilcoxon_df = pd.read_csv(wilcoxon_path)

    cols = [
        c for c in [
            "model_a",
            "model_b",
            "n_matched_units",
            "mean_difference_a_minus_b",
            "p_value_raw",
            "p_value_holm",
            "rank_biserial_r",
            "significant_holm_0_05",
        ]
        if c in wilcoxon_df.columns
    ]

    display(
        wilcoxon_df[cols]
        .sort_values("p_value_holm", na_position="last")
    )
else:
    print("Missing:", wilcoxon_path)


### 19.4 Proposed model versus baselines

In [ ]:

if wilcoxon_path.exists():
    proposed_name = "chilli_lite_gfnet"

    proposed_rows = wilcoxon_df[
        (wilcoxon_df["model_a"] == proposed_name)
        | (wilcoxon_df["model_b"] == proposed_name)
    ].copy()

    if proposed_rows.empty:
        print("No proposed-model Wilcoxon rows found.")
    else:
        display(
            proposed_rows.sort_values("p_value_holm", na_position="last")
        )
        proposed_rows.to_csv(
            TABLE_DIR / "proposed_vs_baselines_wilcoxon.csv",
            index=False,
        )


## 20. McNemar and paired-bootstrap outputs

In [ ]:

if STAT_DIR.exists():
    special_files = list(STAT_DIR.glob("mcnemar_exact__*.csv")) + \
                    list(STAT_DIR.glob("paired_bootstrap__*.csv"))

    if not special_files:
        print(
            "No McNemar/bootstrap files found yet.\n"
            "Run statistical_analysis.py with --model-a, --model-b and --bootstrap."
        )
    else:
        for file in sorted(special_files):
            print("\n", file.name)
            display(pd.read_csv(file))



## 21. Generalization-gap analysis

This section compares each model's average within-domain Macro-F1 with its average pairwise cross-dataset Macro-F1.

Because pairwise tasks use different shared label spaces, this is a **high-level robustness summary**, not a strictly identical-task comparison.


In [ ]:

within = (
    summary[summary["experiment"] == "within_cv"]
    .groupby(["model_name", "model_display"])["test_macro_f1"]
    .mean()
    .rename("within_macro_f1")
)

cross = (
    summary[summary["experiment"] == "pairwise"]
    .groupby(["model_name", "model_display"])["test_macro_f1"]
    .mean()
    .rename("cross_macro_f1")
)

gap = pd.concat([within, cross], axis=1).dropna().reset_index()
gap["generalization_gap"] = gap["within_macro_f1"] - gap["cross_macro_f1"]
gap = gap.sort_values("generalization_gap")

if gap.empty:
    print("Need both within_cv and pairwise results.")
else:
    display(gap)
    gap.to_csv(TABLE_DIR / "generalization_gap.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(gap["model_display"], gap["generalization_gap"])
    ax.set_xlabel("Within-domain Macro-F1 − Cross-dataset Macro-F1")
    ax.set_title("Generalization gap (lower is better)")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    path = FIGURE_DIR / "generalization_gap.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)
    plt.show()



## 22. Export compact LaTeX tables

These exports can be inserted into the manuscript with `\input{...}` after review.


In [ ]:

def export_latex_table(df, filename, index=False, float_format="%.4f"):
    path = TABLE_DIR / filename
    latex = df.to_latex(
        index=index,
        escape=True,
        float_format=float_format,
    )
    path.write_text(latex, encoding="utf-8")
    print("Saved:", path)


if not pairwise_table.empty:
    export_latex_table(
        pairwise_table,
        "pairwise_cross_dataset_results.tex",
        index=False,
    )

if not multisource_table.empty:
    export_latex_table(
        multisource_table,
        "multisource_results.tex",
        index=False,
    )

if not within_table.empty:
    export_latex_table(
        within_table,
        "within_cv_results.tex",
        index=False,
    )

if not pooled_table.empty:
    export_latex_table(
        pooled_table,
        "pooled_cv_results.tex",
        index=False,
    )



## 23. Final manuscript-ready checklist

Before copying tables or figures into the paper:

1. Confirm all intended models completed the same scenarios.
2. Confirm final seed counts are identical for paired statistical comparisons.
3. Report mean ± SD for repeated folds/seeds.
4. Use Macro-F1 and MCC as primary robustness metrics.
5. Use Holm-adjusted \(p\)-values for multiple comparisons.
6. Report effect sizes together with significance.
7. For the proposed model versus the strongest baseline, include:
   - exact McNemar test;
   - paired-bootstrap 95% CI.
8. Keep `exact_target_overlap_removed` in the audit trail.
9. State that SHA-256 detects exact duplicates only, not transformed near-duplicates.
10. Do not interpret pooled CV as external validation.


In [ ]:

print("Analysis complete.")
print("Manuscript figures:", FIGURE_DIR)
print("Manuscript tables :", TABLE_DIR)
